In [ ]:
!apt-get install -y ffmpeg
!pip install -q pandas openpyxl librosa tqdm requests

import os
import zipfile
import requests
import pandas as pd
import librosa
from tqdm import tqdm
import re


YANDEX_ZIP_URL = "YANDEX DISK LINK"
EXCEL_PATH = "input.xlsx"
WORK_DIR = "data"
EXTRACT_DIR = os.path.join(WORK_DIR, "wav")

os.makedirs(EXTRACT_DIR, exist_ok=True)


def download_yandex(url, output_path="archive.zip"):
    base_url = "https://cloud-api.yandex.net/v1/disk/public/resources/download?public_key="
    final_url = base_url + url
    response = requests.get(final_url).json()
    download_url = response["href"]

    with requests.get(download_url, stream=True) as r:
        with open(output_path, "wb") as f:
            for chunk in r.iter_content(1024):
                f.write(chunk)

download_yandex(YANDEX_ZIP_URL)

with zipfile.ZipFile("archive.zip", 'r') as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)

df = pd.read_excel(EXCEL_PATH, sheet_name="Urmi unlabeled")

df = df[df["audio file"].notna()].copy()

def extract_audio_id(filename):
    match = re.search(r'(audio\d+)_urmi_unlabeled', filename)
    return match.group(1) if match else None

df["audio_id"] = df["audio file"].apply(extract_audio_id)

rows = []

wav_files = []
for root, _, files in os.walk(EXTRACT_DIR):
    for f in files:
        if f.endswith(".wav"):
            wav_files.append(os.path.join(root, f))

for filepath in tqdm(wav_files):
    filename = os.path.basename(filepath)

    match = re.search(r'(audio\d+)_urmi_unlabeled_(\d+)', filename)
    if not match:
        continue

    audio_id = match.group(1)
    part_num = str(int(match.group(2)) + 1)

    # новое имя
    new_name = f"{audio_id}-{part_num}_urmi_unlabeled.wav"
    new_path = os.path.join(EXTRACT_DIR, new_name)

    os.rename(filepath, new_path)

    # длина аудио
    try:
        y, sr = librosa.load(new_path, sr=None)
        duration = librosa.get_duration(y=y, sr=sr)
    except Exception as e:
        print(f"Ошибка в файле: {new_name} | Причина: {e}")
        duration = 0

    total_seconds = int(round(duration))
    h = total_seconds // 3600
    m = (total_seconds % 3600) // 60
    s = total_seconds % 60

    duration_str = f"{h}:{m:02d}:{s:02d}"

    # ищем в таблице
    match_df = df[df["audio_id"] == audio_id]

    if len(match_df) == 0:
        continue

    for _, row in match_df.iterrows():
        rows.append({
            "title of the text": row["title of the text"],
            "audio file": row["audio file"],
            "audio file length": row["audio file length"],
            "source": row["source"],
            "dialect": row["dialect"],
            "parts": new_name,
            "parts' length": str(duration_str)
        })

existing_audio_ids = set([r["audio file"] for r in rows])

for _, row in df.iterrows():
    if row["audio file"] not in existing_audio_ids:
        rows.append({
            "title of the text": row["title of the text"],
            "audio file": row["audio file"],
            "audio file length": row["audio file length"],
            "source": row["source"],
            "dialect": row["dialect"],
            "parts": "no parts",
            "parts' length": "0:00:00"
        })


result_df = pd.DataFrame(rows)


def sort_key(audio_file):
    match = re.search(r'audio(\d+)-(\d+)', audio_file)
    if match:
        return (int(match.group(1)), int(match.group(2)))
    return (999999, 999999)

result_df["sort_key"] = result_df["parts"].apply(sort_key)
result_df = result_df.sort_values(by="sort_key").drop(columns=["sort_key"])

result_df.to_excel("result.xlsx", index=False)

print("Готово! result.xlsx сохранён")